# Sesión 0.6: Diccionarios, `defaultdict` y Contadores de Frecuencia
## Diplomado en Ciencia de Datos e Inteligencia Artificial - INT210

---

### 1. La Analogía Infantil: El Cofre de Llaves Mágicas y la Libreta de Inventario

Imagina una habitación llena de miles de cajas con tesoros:

- **El Diccionario (`dict`):** Tienes un cofre con llaves etiquetadas con nombres únicos (ej. "oro", "diamantes", "mapa"). Cada llave abre inmediatamente su caja correspondiente sin tener que probarla en todas las demás cajas. Pero si intentas usar una llave que no existe, suena una alarma (`KeyError`).
- **El `defaultdict`:** Es una libreta mágica de inventario. Si preguntas cuántas manzanas hay en una caja que nunca habías registrado, en lugar de sonar la alarma, la libreta anota automáticamente la palabra "manzanas", coloca un contador en $0$ y te permite seguir trabajando sin interrupciones.

En Machine Learning y Procesamiento de Lenguaje Natural (NLP), los diccionarios permiten contar millones de palabras y perfiles de usuario a velocidad instantánea.

---
### 2. Rigor Científico: Tablas Hash y Conteo de Frecuencias (Joel Grus, Cap. 2)

Un diccionario en Python es una tabla hash indexada donde la búsqueda, inserción y eliminación se ejecutan en **tiempo constante promedio $\mathcal{O}(1)$**.

Al procesar texto no estructurado o flujos de eventos de red, se requiere contar ocurrencias de elementos discretos. Evaluamos tres técnicas progresivas:
1. **Diccionario tradicional con `in` o `.get()`:** `conteo[palabra] = conteo.get(palabra, 0) + 1`
2. **`collections.defaultdict(int)`:** Inicializa automáticamente cualquier clave inexistente con el valor de retorno de `int()` (cero).
3. **`collections.Counter`:** Estructura especializada de alto rendimiento en C que calcula frecuencias e implementa `.most_common(k)`.

In [ ]:
from collections import defaultdict, Counter

# Texto real de prueba: Muestra de correos electrónicos de Phishing y Seguridad
corpus_correos = [
    "URGENT verify your bank account now click link to claim reward",
    "Warning unauthorized security alert on your account login immediately",
    "Your urgent reward is waiting verify account identity claim cash prize",
    "Team meeting scheduled for tomorrow project sprint review"
]

# Método 1: Conteo con diccionario nativo y .get()
conteo_manual = {}
for correo in corpus_correos:
    for palabra in correo.lower().split():
        conteo_manual[palabra] = conteo_manual.get(palabra, 0) + 1

# Método 2: Conteo optimizado con defaultdict(int)
conteo_default = defaultdict(int)
for correo in corpus_correos:
    for palabra in correo.lower().split():
        conteo_default[palabra] += 1

# Método 3: Conteo declarativo de alto rendimiento con Counter
todas_las_palabras = [p for correo in corpus_correos for p in correo.lower().split()]
contador_frecuencias = Counter(todas_las_palabras)

print("=== Las 5 Palabras Más Frecuentes en el Corpus (Counter) ===")
for palabra, freq in contador_frecuencias.most_common(5):
    print(f"Término: '{palabra:15s}' | Frecuencia: {freq:2d}")

#### Explicación Técnica Línea por Línea del Bloque de Código:

- `from collections import defaultdict, Counter`: Importa del módulo estándar de CPython las estructuras de datos optimizadas para conteo asociativo.
- `conteo_manual.get(palabra, 0)`: Busca la clave `palabra` en la tabla hash. Si existe, retorna su valor actual; si no existe, retorna el valor por defecto `0`, evitando un `KeyError`.
- `conteo_default = defaultdict(int)`: Crea un diccionario cuya fábrica por defecto (*default factory*) es `int`. Al acceder a una clave nueva, se ejecuta `int()` (que retorna `0`) y se almacena en memoria de inmediato.
- `[p for correo in corpus_correos for p in correo.lower().split()]`: Comprensión anidada que aplana (*flattens*) el corpus en una lista unidimensional de tokens en minúsculas.
- `contador_frecuencias.most_common(5)`: Utiliza internamente un montículo (*Max-Heap*) para extraer los 5 elementos de mayor frecuencia en tiempo $\mathcal{O}(N \log k)$.

---
### 3. Caso de Estudio 1: Perfilador de Densidad de Palabras Sospechosas en Ciberseguridad

Calcula la densidad de términos de ingeniería social (*Social Engineering Score*) en un mensaje nuevo comparando sus palabras con un diccionario de pesos de riesgo.

In [ ]:
# Diccionario de pesos de riesgo por palabra clave (Threat Intelligence Lexicon)
PESOS_RIESGO = {
    'urgent': 3.0,
    'verify': 2.5,
    'account': 2.0,
    'claim': 3.0,
    'reward': 2.5,
    'login': 1.5,
    'bank': 2.0,
    'click': 1.5
}

def calcular_puntaje_phishing(texto_mensaje: str) -> float:
    """
    Calcula la suma acumulada de riesgo léxico de un mensaje.
    """
    tokens = texto_mensaje.lower().split()
    conteo_mensaje = Counter(tokens)
    
    puntaje_total = sum(
        conteo_mensaje[palabra] * PESOS_RIESGO.get(palabra, 0.0)
        for palabra in conteo_mensaje
    )
    return puntaje_total

mensaje_sospechoso = "Urgent action required: verify your bank account and claim reward"
mensaje_normal = "Please review the quarterly financial sprint report before the meeting"

print(f"Puntaje de Riesgo (Sospechoso): {calcular_puntaje_phishing(mensaje_sospechoso):.2f}")
print(f"Puntaje de Riesgo (Normal):      {calcular_puntaje_phishing(mensaje_normal):.2f}")

#### Explicación Técnica Línea por Línea:
- `PESOS_RIESGO = {...}`: Diccionario hash que almacena los coeficientes de amenaza calculados por analistas de seguridad.
- `conteo_mensaje = Counter(tokens)`: Tokeniza y cuenta la multiplicidad de cada palabra en el mensaje analizado en $\mathcal{O}(L)$, donde $L$ es la longitud del texto.
- `PESOS_RIESGO.get(palabra, 0.0)`: Asigna un peso de $0.0$ a las palabras comunes (artículos, preposiciones) sin generar excepciones.

---
### 4. Caso de Estudio 2: Agrupación de Accesos de Red con `defaultdict(list)`

Agrupa un flujo de conexiones de red mapeando cada dirección IP a una lista de puertos a los que ha intentado acceder.

In [ ]:
# Registro de eventos de red
eventos_red = [
    ('192.168.1.50', 80),
    ('198.51.100.4', 22),
    ('192.168.1.50', 443),
    ('198.51.100.4', 23),
    ('198.51.100.4', 445),
    ('10.0.0.12', 8080)
]

mapa_puertos_por_ip = defaultdict(list)
for ip, puerto in eventos_red:
    mapa_puertos_por_ip[ip].append(puerto)

print("=== Mapeo de Puertos Accedidos por Dirección IP ===")
for ip, lista_puertos in mapa_puertos_por_ip.items():
    print(f"IP: {ip:15s} -> Puertos: {lista_puertos} (Total: {len(lista_puertos)})")

#### Explicación Técnica Línea por Línea:
- `mapa_puertos_por_ip = defaultdict(list)`: Configura una fábrica que instancia una nueva lista vacía `[]` la primera vez que se observa una nueva IP.
- `mapa_puertos_por_ip[ip].append(puerto)`: Añade el puerto de destino directamente a la lista de esa IP sin necesidad de comprobar `if ip not in mapa:`.

---
### 5. Ejercicio Práctico Guiado: Detección de Barrido de Puertos (*Port Scan*) (`# TODO`)

**Instrucción:** Completa la función `detectar_escaneo_puertos` para que identifique a cualquier IP que haya intentado acceder a $\ge 3$ puertos distintos.

In [ ]:
def detectar_escaneo_puertos(eventos: list) -> list:
    """
    Recibe una lista de tuplas (ip, puerto) y retorna la lista de IPs sospechosas de Port Scanning.
    """
    ### TU CÓDIGO AQUÍ
    pass

# Comprobación del ejercicio
# atacantes = detectar_escaneo_puertos(eventos_red)
# print(f"IPs identificadas como atacantes: {atacantes}")  # Debe ser ['198.51.100.4']